<a href="https://colab.research.google.com/github/iking919/Projects-in-AI-and-ML/blob/main/HW6_Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projects in ML & AI HW 6 Part 1 - Izaak King

## Part 1: Reinforcement Learning

In this task, we implement value iteration, a reinforcement learning algorithm on a small discrete Markov Decision Process (MDP). We use a simple Grid World environment, which will be used to illustrate how agents learn optimal behavior through rewards and transitions. The goal is to compute the optimal value function and derive the corresponding optimal policy.

### MDP Formulation

**State Space**:

*  We define the Grid World as a 4x4 grid
*  Each cell represents a state, so the state space consists of 16 discrete states



**Action Space**:

*   The agent can take one of four directional actions at each state
*   Our action space is up, down, left, and right


**Reward Structure**:

*   Each step incurs a small penalty of -1, encouraging the agent to find shorter paths

*   Reaching the terminal state yields a reward of 0.

*   The terminal state is located at the bottom right corner of the grid.

*   Any action that would move the agent off the grid instead leaves it in the same state.

**Discount Factor** :

*  We use a discount factor γ=0.9, which balances immediate and future rewards


<br>

This setup satisfies the Markov property because the next state and reward depend only on the current state and action, not on the history of past states

### Implementation of Value Iteration

Value iteration iteratively updates the value function using the Bellman optimality equation. We repeatedly update the value function until convergence.

In [ ]:
import numpy as np

# Grid size
n = 4
states = [(i, j) for i in range(n) for j in range(n)]
actions = ['up', 'down', 'left', 'right']

# Parameters
gamma = 0.9
theta = 1e-4

# Initialize value function
V = np.zeros((n, n))

# Terminal state
terminal = (3, 3)

# Transition function
def step(state, action):
    if state == terminal:
        return state, 0
    i, j = state
    if action == 'up':
        i = max(i-1, 0)
    elif action == 'down':
        i = min(i+1, n-1)
    elif action == 'left':
        j = max(j-1, 0)
    elif action == 'right':
        j = min(j+1, n-1)
    next_state = (i, j)
    reward = -1
    return next_state, reward

# Value Iteration
while True:
    delta = 0
    new_V = np.copy(V)
    for s in states:
        if s == terminal:
            continue
        values = []
        for a in actions:
            (ni, nj), r = step(s, a)
            values.append(r + gamma * V[ni, nj])
        new_V[s] = max(values)
        delta = max(delta, abs(new_V[s] - V[s]))
    V = new_V
    if delta < theta:
        break

print("Value Function:")
print(V)

Value Function:
[[-4.68559 -4.0951  -3.439   -2.71   ]
 [-4.0951  -3.439   -2.71    -1.9    ]
 [-3.439   -2.71    -1.9     -1.     ]
 [-2.71    -1.9     -1.       0.     ]]


The value function shows the expected cumulative reward from each state when following the optimal policy. States closer to the terminal state have higher (less negative) values, while states farther away have lower values due to accumulating more step penalties of -1. The terminal state has a value of 0, and the values increase as the agent moves closer to it, reflecting that fewer steps are needed to reach the goal.

### Computing the Final Policy

After computing the value function, we derive the optimal policy by selecting the action that maximizes the expected return based on the learned values for each state. This is done by evaluating all possible actions using the value function and choosing the one with the highest value. The resulting policy tells the agent the best action to take from each state to reach the terminal state as efficiently as possible.

In [ ]:
# Compute the optimal policy
policy = np.empty((n, n), dtype=object)

for s in states:
    if s == terminal:
        policy[s] = 'T'  # mark terminal
        continue

    best_action = None
    best_value = -float('inf')

    for a in actions:
        (ni, nj), r = step(s, a)
        val = r + gamma * V[ni, nj]

        if val > best_value:
            best_value = val
            best_action = a

    policy[s] = best_action

print("\nOptimal Policy:")
print(policy)


Optimal Policy:
[['down' 'down' 'down' 'down']
 ['down' 'down' 'down' 'down']
 ['down' 'down' 'down' 'down']
 ['right' 'right' 'right' 'T']]


The optimal policy shows that from nearly every state, the agent chooses to move downward until it reaches the bottom row of the grid. Once on the bottom row, the agent moves right toward the terminal state located at the bottom-right corner. The terminal state is marked with T. This policy reflects the goal of minimizing the number of steps needed to reach the terminal state, since each move incurs a reward of -1.

### Changing the Discount Factor

To understand the impact of the discount factor, we run an experiment by rerunning value iteration with a smaller
γ=0.5. A lower discount factor makes the agent prioritize immediate rewards more strongly.

In this case, we expect the values become less sensitive to distant rewards, and the policy may become more short-sighted, preferring actions that reduce immediate penalties rather than planning long-term paths.

In [ ]:
gamma = 0.5
V = np.zeros((n, n))

while True:
    delta = 0
    new_V = np.copy(V)
    for s in states:
        if s == terminal:
            continue
        values = []
        for a in actions:
            (ni, nj), r = step(s, a)
            values.append(r + gamma * V[ni, nj])
        new_V[s] = max(values)
        delta = max(delta, abs(new_V[s] - V[s]))
    V = new_V
    if delta < theta:
        break

print("\nValue Function with gamma=0.5:")
print(V)


Value Function with gamma=0.5:
[[-1.96875 -1.9375  -1.875   -1.75   ]
 [-1.9375  -1.875   -1.75    -1.5    ]
 [-1.875   -1.75    -1.5     -1.     ]
 [-1.75    -1.5     -1.       0.     ]]


### Impact of Discount Factor on Reward Preference

We experiment by modifying the original Grid World environment, to better understand how the discount factor
γ affects an agent's behavior when faced with competing reward options. Instead of having a single terminal state, we introduce two terminal states. One is closer but offers a smaller reward, and another is farther away but offers a larger reward. Specifically, the agent starts at position (0,0) with a nearby terminal state at (1,1) giving a reward of +2, and a distant terminal state at (3,3) giving a reward of +10. Additionally, we remove the step penalty so that the agent's decisions are influenced purely by the reward values and not by movement cost.

This setup allows us to isolate and observe the effect of the discount factor on decision-making. By running value iteration with different values of γ, we can see how the agent balances immediate versus future rewards. A smaller γ encourages the agent to prioritize rewards that are closer in time, while a larger γ makes the agent more willing to pursue larger rewards that require more steps to obtain.

Through this experiment, we gain insight into how the choice of discount factor can significantly impact the learned policy in reinforcement learning.

In [ ]:
def run_experiment(gamma):
    V = np.zeros((n, n))
    terminal_states = {(1, 1): 2, (3, 3): 10}

    def new_step(state, action):
        if state in terminal_states:
            return state, 0 # No further rewards after reaching terminal

        i, j = state
        if action == 'up': i = max(i-1, 0)
        elif action == 'down': i = min(i+1, n-1)
        elif action == 'left': j = max(j-1, 0)
        elif action == 'right': j = min(j+1, n-1)

        next_state = (i, j)
        # Give terminal reward if moving into terminal state, else 0
        reward = terminal_states.get(next_state, 0)
        return next_state, reward

    # Value Iteration
    while True:
        delta = 0
        new_V = np.copy(V)
        for s in states:
            if s in terminal_states:
                new_V[s] = terminal_states[s] # Value of terminal state is its reward
                continue
            values = []
            for a in actions:
                ns, r = new_step(s, a)
                values.append(r + gamma * V[ns[0], ns[1]])
            new_V[s] = max(values)
            delta = max(delta, abs(new_V[s] - V[s]))
        V = new_V
        if delta < theta:
            break

    # Policy Extraction
    policy = np.empty((n, n), dtype=object)
    for s in states:
        if s in terminal_states:
            policy[s] = 'T'
            continue
        best_action, best_value = None, -float('inf')
        for a in actions:
            ns, r = new_step(s, a)
            val = r + gamma * V[ns[0], ns[1]]
            if val > best_value:
                best_value = val
                best_action = a
        policy[s] = best_action

    return policy

print("Policy with gamma = 0.2 (Short-sighted):")
print(run_experiment(0.2))

print("\nPolicy with gamma = 0.9 (Far-sighted):")
print(run_experiment(0.9))

Policy with gamma = 0.2 (Short-sighted):
[['down' 'down' 'down' 'down']
 ['right' 'T' 'left' 'down']
 ['up' 'up' 'down' 'down']
 ['right' 'right' 'right' 'T']]

Policy with gamma = 0.9 (Far-sighted):
[['down' 'right' 'down' 'down']
 ['down' 'T' 'down' 'down']
 ['down' 'down' 'down' 'down']
 ['right' 'right' 'right' 'T']]


The results of this experiment clearly show how the discount factor
γ affects the agent's decision-making. When γ=0.2, the agent is more short-sighted, meaning it heavily discounts future rewards. As a result, the policy tends to direct the agent toward the closer terminal state at (1,1) with reward +2. This is reflected in the policy where many states guide the agent upward or leftward toward the nearby goal, even though a larger reward exists farther away. The agent prioritizes reaching a reward quickly rather than maximizing the total reward.

In contrast, when γ=0.9, the agent becomes more far-sighted and places greater value on future rewards. The policy now more consistently directs the agent toward the farther terminal state at (3,3) with reward +10. Even though this path requires more steps, the higher reward is worth pursuing when future rewards are not heavily discounted. This demonstrates that increasing the discount factor encourages the agent to favor long-term gains over immediate, smaller rewards, significantly altering the learned policy.

### Discussion:

This setup is a Markov Decision Process (MDP) because it satisfies all the required components. It has a discrete state space (all grid positions), a set of actions (up, down, left, right), a transition model (deterministic movement based on actions), a reward function (step penalties or terminal rewards), and a discount factor that balances immediate and future rewards. The Markov property is satisfied because the next state and received reward depend only on the current state and action, not on the history of previous states.

The learned policy tells the agent which action to take from each state to maximize cumulative reward. In the first experiment with a single terminal state, the policy guides the agent along the shortest path to minimize penalties. In the second experiment with multiple terminal states, the policy changes depending on the discount factor. A low discount factor favors the closer, smaller reward, while a high discount factor favors the farther, larger reward. This illustrates the trade-off between immediate and long-term rewards in reinforcement learning.